# PATH MANAGEMENT

In [1]:
import os

print(os.getcwd())
if not os.getcwd().endswith("app"):
    os.chdir("../app")
    print(os.getcwd())

import pandas as pd
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)

%load_ext autoreload
%autoreload 2
# %matplotlib inline

/home/turbotowerlnx/Documents/Master/TA/TA-Spanish-Esperanto-Translator/notebooks
/home/turbotowerlnx/Documents/Master/TA/TA-Spanish-Esperanto-Translator/app


In [2]:
from src.config import Configuration

CONFIG = Configuration(
    batch_size=8, # NOTE: CHANGE    
    max_tok_length=64, # NOTE: CHANGE
    max_epoch=10, # NOTE: CHANGE

    model_name="google/mt5-large"

    
)

# Fine-tuning

Fine-tuning refers to the process in transfer learning in which the parameter values of a model trained on a large dataset are modified when the training process continues on a small dataset (see [Kevin Murphy's book](https://probml.github.io/pml-book/book1.html) Section 19.2 for further details). The main motivation is to adapt a pre-trained model trained on a large amount of data to tackle a specific task providing better performance that would be achieved training on the small task-specific dataset.

In this notebook, we are going to use for fine-tuning a dataset set that is already available in the [Datasets repository](https://huggingface.co/datasets) from Hugging Face. However, the [Datasets library](https://huggingface.co/docs/datasets) makes easy to access and load datasets. For example, you can easily load your own dataset following [this tutorial](https://huggingface.co/docs/datasets/loading#local-and-remote-files).

More precisely, we are going to explain how to fine-tune the [NLLB model](https://huggingface.co/docs/transformers/model_doc/nllb) on the [Europarl-ST dataset](https://huggingface.co/datasets/tj-solergibert/Europarl-ST), but only that [dataset of Europarl-ST focused on the text data for MT from English](https://huggingface.co/datasets/tj-solergibert/Europarl-ST-processed-mt-en).

In [3]:
# from datasets import load_dataset

# raw_datasets = load_dataset("tj-solergibert/Europarl-ST-processed-mt-en")

# print(raw_datasets)

from src.data import get_es_eo_dataset

CONFIG.corpus_path = CONFIG.corpus_path_smaller  # NOTE: CHANGE
raw_datasets = get_es_eo_dataset(CONFIG)

print(raw_datasets)

/home/turbotowerlnx/Documents/Master/TA/TA-Spanish-Esperanto-Translator/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DatasetDict({
    train: Dataset({
        features: ['source_text', 'dest_text', 'dest_lang'],
        num_rows: 20097
    })
    test: Dataset({
        features: ['source_text', 'dest_text', 'dest_lang'],
        num_rows: 4306
    })
    valid: Dataset({
        features: ['source_text', 'dest_text', 'dest_lang'],
        num_rows: 4306
    })
})


As shown, the Europarl-ST already comes with a pre-defined partition on the three conventional sets: training, validation and test. Each set is a dictionary with a list of source sentences (source_text), target sentences (dest_text) and the target language (dest_lang).

Let's take a closer look at the features of the training set:

In [4]:
raw_datasets["train"].features

{'source_text': Value('string'),
 'dest_text': Value('string'),
 'dest_lang': Value('int64')}

As you can see, the possible target languages are German, English, Spanish, French, Italian, Dutch, Polish, Portuguese and Romanian.

Let us take a look at the translations of the first two English sentences:

In [5]:
raw_datasets["train"][:14]["source_text"]

['luego se puso mi padre.',
 'para lo mejor o para lo peor',
 'zai jiam (creo que era así)',
 'el sol le vino de perlas para su cuidada melena rubia.',
 'considerando que las operaciones tales como angelfire y geo cities? han existido desde los primeros días de la web, las nuevas ofertas de, por ejemplo, facebook y my space? actualmente tiene muchos seguidores.',
 'sí, todos saben que es uno de los mejores centrocampistas atacantes ingleses de su generación.',
 'luego compiten en una batalla de baile contra tres jóvenes que visten ropa deportiva azul marca adidas.',
 'él tradujo algunos de mis poemas.',
 'museo manuel lópez villaseñor: contiene la obra del ciudadrealeño manuel lópez-villaseñor, uno de los máximos exponentes de la pintura española de la segunda mitad del siglo xx.',
 '¿quién es este ser que nos estás mostrando?',
 '(la película no debe tener lugar donde esté la cámara, el rodaje debe tener lugar donde la película tiene lugar).4.',
 'el pronombre de tercera persona neutr

In [6]:

raw_datasets["train"][:14]["dest_text"]

['tiam mia patro sidiĝis.',
 'por pli bone aŭ for worse',
 'cao zhi (caŭ ĝi)',
 "parfumoj preparitaj por ŝia blonda har'.",
 'dum operacioj kiel ekzemple angelfire kaj geocities ekzistis ekde la fruaj tagoj de la reto, pli novaj proponoj de, ekzemple, facebook kaj twitter nuntempe havas grandajn sekvantajn.',
 'jes, ĉiuj scias, ke li estas unu el la plej bonaj anglaj atakantoj de sia generacio.',
 'poste ili danckonkursas kontraŭ tri junaj viroj kiuj estas vestitaj en bluaj sportvestoj de la marko adidas.',
 'li tradukis miajn poemojn',
 'muzeo manuel lópez-villaseñor: enhavas verkaro de la ciudadreala pentristo manuel lópez-villaseñor, nome unu el la ĉefaj reprezentantoj de la hispana realisma pentrarto de la dua duono de la 20a jarcento.',
 'kies letero estas tiu, kiun vi montris al mi?',
 '(la filmo ne okazu kie la kamerao staras; filmado okazu kie la filmo okazas.)',
 'sekse neŭtrala pronomo estas "ĝi".',
 'eĉ kiam la suno ne videblas.',
 'tion nur mi volas diri,']

In [7]:
raw_datasets["train"][:14]["dest_lang"]

[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

As shown, each English sentence is repeated for each of the seven target languages (0: 'de', 2: 'es', 3: 'fr', 4: 'it', 5: 'nl', 6: 'pl', 7: 'pt').

Provided that the NLLB model was pretrained on sentence pairs involving 200 languages, being one of the them the translation from English into Spanish, we are going to be filtering Europarl-ST only for English into Spanish using a simple [lambda function](https://realpython.com/python-lambda/) with the [Dataset.filter() function](https://huggingface.co/docs/datasets/v2.9.0/en/package_reference/main_classes#datasets.Dataset.filter).

In [8]:
# lang="es"
# lang_id = raw_datasets["train"].features["dest_lang"].names.index(lang)
# raw_datasets = raw_datasets.filter(lambda x: x["dest_lang"] == lang_id)

Now we load the pre-trained tokenizer for the NLLB model and apply it to the English-Spanish pair:

In [9]:
from transformers import AutoTokenizer

checkpoint = CONFIG.model_name
# mT5 doesn't use src_lang/tgt_lang parameters like NLLB
# It uses task prefixes instead
tokenizer = AutoTokenizer.from_pretrained(
    checkpoint, 
    padding=True, 
    pad_to_multiple_of=8, 
    truncation=True, 
    max_length=CONFIG.max_tok_length,
)

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


/home/turbotowerlnx/Documents/Master/TA/TA-Spanish-Esperanto-Translator/venv/lib/python3.12/site-packages/transformers/convert_slow_tokenizer.py:564: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


We can apply the tokenizer function to any dataset taking advantage that Hugging Face Datasets are [Apache Arrow](https://arrow.apache.org) files stored on the disk, so you only keep the samples you ask for loaded in memory.

To keep the data as a dataset, we will use the [Dataset.map() function](https://huggingface.co/docs/datasets/en/package_reference/main_classes#datasets.Dataset.map). This also allows us some extra flexibility, if we need more preprocessing done than just tokenization. The map() method works by applying a function on each element of the dataset.

In our case, each sample pair is going to be preprocessed according to the training needs of the model that is to be finetuned:

In [10]:
def preprocess_function(sample):
    # mT5 requires task prefix prepended to source text
    prefixed_sources = [CONFIG.task_prefix + text for text in sample["source_text"]]
    
    model_inputs = tokenizer(
        prefixed_sources, 
        text_target = sample["dest_text"],

        )
    return model_inputs

The way the Datasets library applies this processing is by adding new fields to the datasets, one for each key in the dictionary returned by the tokenize function, that is, *input_ids*, *attention_mask* and *labels*. We can check what the preprocess_function is doing with a small sample

In [11]:
sample = raw_datasets["train"].select(range(2))
model_input = preprocess_function({
    "source_text": list(sample["source_text"]),
    "dest_text": list(sample["dest_text"]),
})
print(model_input)

{'input_ids': [[37194, 702, 259, 29037, 288, 26609, 267, 1411, 1796, 303, 259, 55110, 658, 25160, 260, 1], [37194, 702, 259, 29037, 288, 26609, 267, 435, 707, 5541, 259, 268, 435, 707, 603, 723, 1]], 'attention_mask': [[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]], 'labels': [[259, 21837, 11174, 86504, 68146, 23311, 260, 1], [519, 6744, 35144, 11718, 332, 48461, 265, 1]]}


In [12]:
for sample in model_input['input_ids']:
    print(tokenizer.convert_ids_to_tokens(sample))

['▁translate', '▁from', '▁', 'Spanish', '▁to', '▁Esperanto', ':', '▁lu', 'ego', '▁se', '▁', 'puso', '▁mi', '▁padre', '.', '</s>']
['▁translate', '▁from', '▁', 'Spanish', '▁to', '▁Esperanto', ':', '▁para', '▁lo', '▁mejor', '▁', 'o', '▁para', '▁lo', '▁pe', 'or', '</s>']


We can recover the source text by applying [batch_decode](https://huggingface.co/docs/transformers/en/internal/tokenization_utils#transformers.PreTrainedTokenizerBase.batch_decode) of the tokenizer 

In [13]:
tokenizer.batch_decode(model_input['input_ids'])

['translate from Spanish to Esperanto: luego se puso mi padre.</s>',
 'translate from Spanish to Esperanto: para lo mejor o para lo peor</s>']

Now, we can apply the preprocess_function to the raw datasets (training, validation and test):

In [14]:
tokenized_datasets = raw_datasets.map(preprocess_function, batched=True)

Map:   0%|                                                                                                                                          | 0/20097 [00:00<?, ? examples/s]

Map:  25%|██████████████████████████████▊                                                                                             | 5000/20097 [00:00<00:00, 20430.40 examples/s]

Map:  55%|███████████████████████████████████████████████████████████████████▎                                                       | 11000/20097 [00:00<00:00, 32431.17 examples/s]

Map:  85%|████████████████████████████████████████████████████████████████████████████████████████████████████████                   | 17000/20097 [00:00<00:00, 38633.75 examples/s]

Map: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 20097/20097 [00:00<00:00, 29078.91 examples/s]

Map:   0%|                                                                                                                                           | 0/4306 [00:00<?, ? examples/s]

Map: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 4306/4306 [00:00<00:00, 45998.63 examples/s]

Map:   0%|                                                                                                                                           | 0/4306 [00:00<?, ? examples/s]

Map: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 4306/4306 [00:00<00:00, 47309.22 examples/s]

We are going to filter the tokenized datasets by maximum number of tokens in source and target language:

In [15]:
tokenized_datasets = tokenized_datasets.filter(lambda x: len(x["input_ids"]) <= CONFIG.max_tok_length and len(x["labels"]) <= CONFIG.max_tok_length , desc=f"Discarding source and target sentences with more than {CONFIG.max_tok_length} tokens")

Discarding source and target sentences with more than 64 tokens:   0%|                                                                              | 0/20097 [00:00<?, ? examples/s]

Discarding source and target sentences with more than 64 tokens:  40%|█████████████████████████▍                                      | 8000/20097 [00:00<00:00, 62248.85 examples/s]

Discarding source and target sentences with more than 64 tokens:  80%|██████████████████████████████████████████████████▏            | 16000/20097 [00:00<00:00, 63614.56 examples/s]

Discarding source and target sentences with more than 64 tokens: 100%|███████████████████████████████████████████████████████████████| 20097/20097 [00:00<00:00, 63374.28 examples/s]

Discarding source and target sentences with more than 64 tokens:   0%|                                                                               | 0/4306 [00:00<?, ? examples/s]

Discarding source and target sentences with more than 64 tokens: 100%|█████████████████████████████████████████████████████████████████| 4306/4306 [00:00<00:00, 62554.28 examples/s]

Discarding source and target sentences with more than 64 tokens:   0%|                                                                               | 0/4306 [00:00<?, ? examples/s]

Discarding source and target sentences with more than 64 tokens: 100%|█████████████████████████████████████████████████████████████████| 4306/4306 [00:00<00:00, 60990.86 examples/s]

We can take a quick look at the length histogram in the source language:

In [16]:
dic = {}
for sample in tokenized_datasets['train']:
    sample_length = len(sample['input_ids'])
    if sample_length not in dic:
        dic[sample_length] = 1
    else:
        dic[sample_length] += 1 

for i in range(1,CONFIG.max_tok_length+1):
    if i in dic:
        print(f"{i:>2} {dic[i]:>3}")

10   5
11  28
12 139
13 364
14 678
15 949
16 1066
17 1148
18 1143
19 1033
20 987
21 820
22 732
23 659
24 587
25 549
26 457
27 421
28 388
29 352
30 334
31 299
32 285
33 276
34 274
35 211
36 245
37 245
38 208
39 194
40 199
41 178
42 182
43 191
44 162
45 174
46 159
47 160
48 150
49 169
50 152
51 155
52 112
53 135
54 123
55 126
56 119
57 108
58 101
59 102
60  95
61 102
62  74
63  79
64  76


Checking a sample after filtering by maximum number of tokens:

In [17]:
for sample in tokenized_datasets['train'].select(range(5)):
    print(sample['input_ids'])
    print(sample['attention_mask'])
    print(sample['labels'])

[37194, 702, 259, 29037, 288, 26609, 267, 1411, 1796, 303, 259, 55110, 658, 25160, 260, 1]
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
[259, 21837, 11174, 86504, 68146, 23311, 260, 1]
[37194, 702, 259, 29037, 288, 26609, 267, 435, 707, 5541, 259, 268, 435, 707, 603, 723, 1]
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
[519, 6744, 35144, 11718, 332, 48461, 265, 1]
[37194, 702, 259, 29037, 288, 26609, 267, 40075, 1359, 579, 274, 49260, 319, 3415, 259, 7590, 271, 1]
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
[4195, 134397, 274, 297, 14238, 259, 14006, 271, 1]
[37194, 702, 259, 29037, 288, 26609, 267, 362, 3208, 340, 42598, 269, 393, 4276, 435, 517, 40110, 906, 31192, 377, 259, 81724, 260, 1]
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
[51816, 1171, 14510, 13788, 519, 259, 24050, 262, 65656, 262, 588, 277, 260, 1]
[37194, 702, 259, 29037, 288, 26609, 267, 5910, 261, 1746, 259, 155099, 319, 655, 4889, 269, 595, 259, 16679, 7026, 27692, 85

bitsandbytes is a quantization library with a Transformers integration. With this integration, you can quantize a model to 8 or 4-bits and enable many other options by configuring the BitsAndBytesConfig class. For example, you can:

<ul>
<li>set load_in_4bit=True to quantize the model to 4-bits when you load it</li>
<li>set bnb_4bit_quant_type="nf4" to use a special 4-bit data type for weights initialized from a normal distribution</li>
<li>set bnb_4bit_use_double_quant=True to use a nested quantization scheme to quantize the already quantized weights</li>
<li>set bnb_4bit_compute_dtype=torch.bfloat16 to use bfloat16 for faster computation</li>
</ul>


In [18]:
import torch
from transformers import BitsAndBytesConfig

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

Pass the quantization_config to the from_pretrained method.

In [19]:
from transformers import AutoModelForSeq2SeqLM

model = AutoModelForSeq2SeqLM.from_pretrained(
    checkpoint,
    quantization_config=quantization_config
    )


Next, you should call the prepare_model_for_kbit_training() function to preprocess the quantized model for training.

In [20]:
from peft import prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=False, gradient_checkpointing_kwargs={'use_reentrant':False})

[LoRA (Low-Rank Adaptation of Large Language Models)](https://huggingface.co/docs/peft/task_guides/lora_based_methods) is a [parameter-efficient fine-tuning (PEFT)](https://huggingface.co/docs/peft/index) technique that significantly reduces the number of trainable parameters. It works by inserting a smaller number of new weights into the model and only these are trained. This makes training with LoRA much faster, memory-efficient, and produces smaller model weights (a few hundred MBs), which are easier to store and share.

Each PEFT method is defined by a PeftConfig class that stores all the important parameters for building a PeftModel. For example, to train with LoRA, load and create a LoraConfig class and specify the following parameters:

<ul>
<li>task_type: the task to train for (sequence-to-sequence language modeling in this case)</li>
<li>r: the dimension of the low-rank matrices</li>
<li>lora_alpha: the scaling factor for the low-rank matrices</li>
<li>target_modules: determine what set of parameters are adapted</li>
<li>lora_dropout: the dropout probability of the LoRA layers</li>
</ul>

In [21]:
from peft import LoraConfig, get_peft_model

config = LoraConfig(
    task_type="SEQ_2_SEQ_LM",
    r=32, # NOTE: CHANGE
    lora_alpha=64, # NOTE: CHANGE
    # mT5 uses different module names than NLLB
    # For mT5, attention modules are named: q, k, v, o (same as T5)
    target_modules=["q", "k", "v", "o"],
    lora_dropout=0.1, # NOTE: CHANGE
    bias="none",
)

Once LoRA and the quantization are setup, create a quantized PeftModel with the get_peft_model() function. It takes a quantized model and the LoraConfig containing the parameters for how to configure a model for training with LoRA.

In [22]:
lora_model = get_peft_model(model, config)
lora_model.print_trainable_parameters()

trainable params: 18,874,368 || all params: 1,248,455,680 || trainable%: 1.5118


The function that is responsible for putting together samples inside a batch is called a collate function. It is an argument you can pass when you build a DataLoader, the default being a function that will just convert your samples to PyTorch tensors and concatenate them. This is not possible in our case since the inputs we have are not all of the same size. We have deliberately postponed the padding, to only apply it as necessary on each batch and avoid having over-long inputs with a lot of padding.

To do this in practice, we have to define a collate function that will apply the correct amount of padding to the items of the dataset we want to batch together. Fortunately, the Transformers library provides us with such a function via DataCollatorForSeq2Seq that takes a tokenizer when you instantiate it (to know which padding token to use, and whether the model expects padding to be on the left or on the right of the inputs), so we will also need to instantiate the model first to provide it to the collate function:

In [23]:
from transformers import DataCollatorForSeq2Seq
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer, 
    model=lora_model, 
    pad_to_multiple_of=8
    )

## Evaluation

The last thing to define for our Seq2SeqTrainer is how to compute the metrics to evaluate the predictions of our model with respect to references. To this purpose, we use the [Evaluate library](https://huggingface.co/docs/evaluate) which includes the definition of generic and task-specific metrics. In our case, we use the [BLEU metric](https://huggingface.co/spaces/evaluate-metric/bleu), or to be more precise, [sacreBLEU](https://huggingface.co/spaces/evaluate-metric/sacrebleu). You can see a simple example of usage below:

:

In [24]:
from evaluate import load

metric_bleu = load("sacrebleu")
metric_comet = load("comet")
metric_chrf = load("chrf")

/home/turbotowerlnx/Documents/Master/TA/TA-Spanish-Esperanto-Translator/venv/lib/python3.12/site-packages/torchmetrics/utilities/imports.py:23: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


Fetching 5 files:   0%|                                                                                                                                        | 0/5 [00:00<?, ?it/s]

Fetching 5 files: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 5/5 [00:00<00:00, 112750.11it/s]

Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.5.6. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../../../../../.cache/huggingface/hub/models--Unbabel--wmt22-comet-da/snapshots/2760a223ac957f30acfb18c8aa649b01cf1d75f2/checkpoints/model.ckpt`


Encoder model frozen.


/home/turbotowerlnx/Documents/Master/TA/TA-Spanish-Esperanto-Translator/venv/lib/python3.12/site-packages/pytorch_lightning/core/saving.py:195: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']


We need to define a function compute_metrics to compute BLEU scores at each epoch. The example below performs a basic post-processing to decode the predictions into texts:

In [25]:
from torch.nn.utils.rnn import pad_sequence
import numpy as np
def postprocess_text(preds, labels):
    preds = [pred.strip() for pred in preds]
    labels = [[label.strip()] for label in labels]

    return preds, labels

def compute_metrics_train(eval_preds):
    preds, labels = eval_preds

    # Convert to lists if coming from a datasets.Column
    if not isinstance(labels, list):
        labels = list(labels)
        
    if isinstance(preds, tuple):
        preds = preds[0]
    
    # NOTE: CHANGE 
    # Replace invalid token IDs in predictions to prevent OverflowError
    # Clip to valid range and handle any overflow issues
    vocab_size = len(tokenizer)
    preds = np.array(preds)
    preds = np.where(
        (preds < 0) | (preds >= vocab_size) | np.isnan(preds) | np.isinf(preds), 
        tokenizer.pad_token_id, 
        preds
    )
    preds = np.clip(preds, 0, vocab_size - 1).astype(np.int64)
    preds = preds.tolist()
    
    # Decode predictions
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

    # Replace negative ids in the labels as we can't decode them.
    labels = [
        [tokenizer.pad_token_id if j < 0 else j for j in label]
        for label in labels
    ]
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    

    # Some simple post-processing
    decoded_preds, decoded_labels = postprocess_text(decoded_preds, decoded_labels)

    result_chrf = metric_chrf.compute(
        predictions=decoded_preds, 
        references=decoded_labels
    )
    result = {
        "chrf": result_chrf["score"],
    }
    

    prediction_lens = [np.count_nonzero(np.array(pred) != tokenizer.pad_token_id) for pred in preds]
    result["gen_len"] = np.mean(prediction_lens)
    result = {k: round(v, 4) for k, v in result.items()}
    return result


def compute_metrics(preds, labels, sources):

    # Convert to lists if coming from a datasets.Column
    if not isinstance(labels, list):
        labels = list(labels)
        
    if isinstance(preds, tuple):
        preds = preds[0]
    
    # NOTE: CHANGE 
    # Replace invalid token IDs in predictions to prevent OverflowError
    if hasattr(preds, '__iter__') and len(preds) > 0:
        # Check if preds contains tensors
        if hasattr(preds[0], 'tolist'):
            # Pad sequences to same length
            preds_padded = pad_sequence(
                [p if len(p.shape) > 0 else p.unsqueeze(0) for p in preds], 
                batch_first=True, 
                padding_value=tokenizer.pad_token_id
            )
            preds = preds_padded.cpu().numpy()
        else:
            preds = np.array(preds) if not isinstance(preds, np.ndarray) else preds
             
    # Clip to valid range and handle any overflow issues
    vocab_size = len(tokenizer)
    # preds = np.array(preds)
    preds = np.where((preds < 0) | (preds >= vocab_size) | np.isnan(preds) | np.isinf(preds), 
                     tokenizer.pad_token_id, 
                     preds)
    preds = np.clip(preds, 0, vocab_size - 1).astype(np.int64)
    preds = preds.tolist()
    
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

    # Replace negative ids in the labels as we can't decode them.
    labels = [
        [tokenizer.pad_token_id if j < 0 else j for j in label]
        for label in labels
    ]
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    
    # Decode sources
    # decoded_sources = tokenizer.batch_decode(sources, skip_special_tokens=True)

    # Some simple post-processing
    decoded_preds, decoded_labels = postprocess_text(decoded_preds, decoded_labels)

    result_blue = metric_bleu.compute(
        predictions=decoded_preds, 
        references=decoded_labels
    )
    result_comet = metric_comet.compute(
        sources=sources,
        predictions=decoded_preds, 
        references=[label[0] for label in decoded_labels]  # COMET expects flat list, not nested
    )
    result_chrf = metric_chrf.compute(
        predictions=decoded_preds, 
        references=decoded_labels
    )
    result = {
        "bleu": result_blue["score"],
        "comet": result_comet["mean_score"],
        "chrf": result_chrf["score"]
    }

    prediction_lens = [np.count_nonzero(np.array(pred) != tokenizer.pad_token_id) for pred in preds]
    result["gen_len"] = np.mean(prediction_lens)
    result = {k: round(v, 4) for k, v in result.items()}
    return result

## Training

The first step before we can define our [Trainer](https://huggingface.co/docs/transformers/en/main_classes/trainer#trainer) is to define a [Seq2SeqTrainingArguments class](https://huggingface.co/docs/transformers/en/main_classes/trainer#transformers.Seq2SeqTrainingArguments) that will contain all the hyperparameters the Trainer will use for training and evaluation. The only compulsory argument you have to provide is a directory where the trained model will be saved, as well as the checkpoints along the way. For all the rest, you can set them depending on the recommendations from the model developers:

In [26]:
from transformers import Seq2SeqTrainingArguments

args = Seq2SeqTrainingArguments(
    CONFIG.output_model_dir,
    eval_strategy = "epoch",
    learning_rate=1e-4,
    per_device_train_batch_size=CONFIG.batch_size,
    per_device_eval_batch_size=CONFIG.batch_size,
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=CONFIG.max_epoch,
    predict_with_generate=True,

    generation_max_length=CONFIG.max_tok_length,
    generation_num_beams=5, # NOTE: CHANGE
    metric_for_best_model="chrf",   # NOTE: CHANGE # Since you're using chrF
)

Once we have our model, we can define a Trainer by passing it all the objects constructed up to now — the model, the training_args, the training and validation datasets, the tokenizer, the data collator and the compute_metrics function:

In [27]:
from transformers import Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    lora_model,
    args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['valid'],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics_train
)


/tmp/ipykernel_318266/1397975236.py:3: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


To fine-tune the model on our dataset, we just have to call the [train() function](https://huggingface.co/docs/transformers/en/main_classes/trainer#transformers.Trainer.train) of our Trainer:

In [28]:
trainer.train()

Epoch,Training Loss,Validation Loss,Chrf,Gen Len
1,2.507700,1.894236,49.885000,39.618700
2,2.337200,1.842778,51.245500,40.085200
3,2.264900,1.796198,51.763000,40.845800
4,2.198300,1.785828,52.313300,40.868200
5,2.156900,1.770105,52.128500,40.144000
6,2.106100,1.758477,52.573200,40.204900
7,2.072000,1.753750,52.973600,40.632900
8,2.043200,1.739798,52.797200,40.458400
9,2.039200,1.742702,53.004700,40.665300
10,2.019000,1.741686,53.082500,40.665300


TrainOutput(global_step=23080, training_loss=2.2484395699856394, metrics={'train_runtime': 10171.862, 'train_samples_per_second': 18.147, 'train_steps_per_second': 2.269, 'total_flos': 5.746253345164493e+16, 'train_loss': 2.2484395699856394, 'epoch': 10.0})

The training stop because pc got powered down... need to continue training jeje

The model seems to start overfitting when taking a look at the validations, we gona let it finish but me end up with worse results

In [29]:
import os
import glob

# Find the latest checkpoint in the output directory
checkpoint_dirs = glob.glob(os.path.join(CONFIG.output_model_dir, "checkpoint-*"))
if checkpoint_dirs:
    latest_checkpoint = max(checkpoint_dirs, key=lambda x: int(x.split("-")[-1]))
    print(f"Resuming training from checkpoint: {latest_checkpoint}")
    
    trainer.train(resume_from_checkpoint=latest_checkpoint)
else:
    print("No checkpoints found. Starting training from scratch.")
    trainer.train()

Resuming training from checkpoint: ../models/mt5-large-finetuned-es-to-eo/checkpoint-23080


Epoch,Training Loss,Validation Loss


In [30]:
# Save the final model
trainer.save_model(os.path.join(CONFIG.output_model_dir, "final_model"))

In [31]:
# Load the final model
from transformers import AutoModelForSeq2SeqLM
from peft import PeftModel

# Load the base model with quantization
base_model = AutoModelForSeq2SeqLM.from_pretrained(
    checkpoint,
    quantization_config=quantization_config
)

# Load the LoRA adapter weights
model = PeftModel.from_pretrained(
    base_model,
    os.path.join(CONFIG.output_model_dir, "final_model")
)

## Inference

At inference time, it is recommended to use [generate()](https://huggingface.co/docs/transformers/v4.26.1/en/main_classes/text_generation#transformers.GenerationMixin.generate). This method takes care of encoding the input and feeding the encoded hidden states via cross-attention layers to the decoder and auto-regressively generates the decoder output. Check out [this blog post](https://huggingface.co/blog/how-to-generate) to know all the details about generating text with Transformers. There’s also [this blog post](https://huggingface.co/blog/encoder-decoder#encoder-decoder) which explains how generation works in general in encoder-decoder models.

Let us first load the default inference parameters of NLLB: 

In [32]:
from transformers import GenerationConfig

generation_config = GenerationConfig.from_pretrained(
    checkpoint,
)

print(generation_config)

GenerationConfig {
  "decoder_start_token_id": 0,
  "eos_token_id": 1,
  "pad_token_id": 0
}



We prepare the test set in batches to be translated:

In [33]:
batch_tokenized_test = tokenized_datasets['test'].batch(CONFIG.batch_size)

Batching examples:   0%|                                                                | 0/3938 [00:00<?, ? examples/s]

Batching examples:  47%|███████████████████████▎                          | 1832/3938 [00:00<00:00, 18099.72 examples/s]

Batching examples:  93%|██████████████████████████████████████████████▌   | 3672/3938 [00:00<00:00, 18216.23 examples/s]

Batching examples: 100%|██████████████████████████████████████████████████| 3938/3938 [00:00<00:00, 17824.20 examples/s]

Processing in batches to add padding and converting to tensors, then perform inference with num_beams = 1 and do_sample = False, that is, greedy search.

In [34]:
import tqdm
number_of_batches = len(batch_tokenized_test["source_text"])
all_sources = []
output_sequences = []
for i in tqdm.tqdm(range(number_of_batches)):
    all_sources.extend(batch_tokenized_test["source_text"][i])

    inputs = tokenizer(
        batch_tokenized_test["source_text"][i], 
        max_length=CONFIG.max_tok_length, 
        truncation=True, 
        return_tensors="pt", 
        padding=True,
        )
    with torch.no_grad():    
        output_batch = model.generate(
            generation_config=generation_config, 
            input_ids=inputs["input_ids"].cuda(), 
            attention_mask=inputs["attention_mask"].cuda(), 
            forced_bos_token_id=tokenizer.convert_tokens_to_ids(CONFIG.tgt_code), 
            max_length = CONFIG.max_tok_length, 
            num_beams=1, 
            do_sample=False,
            )
    output_sequences.extend(output_batch.cpu())

  0%|                                                                                           | 0/493 [00:00<?, ?it/s]

  0%|▏                                                                                  | 1/493 [00:00<06:44,  1.22it/s]

  0%|▎                                                                                  | 2/493 [00:01<08:17,  1.01s/it]

  1%|▌                                                                                  | 3/493 [00:03<08:27,  1.04s/it]

  1%|▋                                                                                  | 4/493 [00:03<07:14,  1.13it/s]

...

 99%|████████████████████████████████████████████████████████████████████████████████▌| 490/493 [09:58<00:04,  1.35s/it]

100%|████████████████████████████████████████████████████████████████████████████████▋| 491/493 [09:59<00:02,  1.33s/it]

100%|████████████████████████████████████████████████████████████████████████████████▊| 492/493 [10:01<00:01,  1.42s/it]

100%|█████████████████████████████████████████████████████████████████████████████████| 493/493 [10:01<00:00,  1.13s/it]

100%|█████████████████████████████████████████████████████████████████████████████████| 493/493 [10:01<00:00,  1.22s/it]

In [35]:
result = compute_metrics(output_sequences, tokenized_datasets["test"]["labels"], all_sources)
print(f'BLEU score: {result["bleu"]:0.4f}')
print(f'COMET score: {result["comet"]:0.4f}')
print(f'CHRF score: {result["chrf"]:0.4f}')

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.


GPU available: True (cuda), used: True


TPU available: False, using: 0 TPU cores


/home/turbotowerlnx/Documents/Master/TA/TA-Spanish-Esperanto-Translator/venv/lib/python3.12/site-packages/torch/__init__.py:1551: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:80.)
  return _C._get_float32_matmul_precision()
You are using a CUDA device ('NVIDIA GeForce RTX 5070') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/t

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


BLEU score: 20.7225
COMET score: 0.7221
CHRF score: 47.1243


In [36]:
from maikol_utils.print_utils import print_separator

# all_sources already contains raw text strings
# output_sequences contains token IDs that need to be decoded
decoded_outputs = tokenizer.batch_decode(output_sequences, skip_special_tokens=True)

# Get reference translations from test set
test_references = tokenized_datasets["test"]["dest_text"]

# Print first 10 examples
for i, (source, output, reference) in enumerate(zip(all_sources[:10], decoded_outputs[:10], test_references[:10])):
    print_separator(f"Example {i+1}:")
    print(f"Source:      {source}")
    print(f"Translation: {output}")
    print(f"Reference:   {reference}")

________________________________________________________________
                           Example 1:                           

Source:      usted, amigo oyente, no está muy
Translation: , amiko aŭskultanto, ne estas tiel granda.
Reference:   cxu vi, amiko, ne tre povas
________________________________________________________________
                           Example 2:                           

Source:      doctor en filología hispanica.
Translation: estas doktoro en hispana lingvo.
Reference:   magistro pri hispana filologio.
________________________________________________________________
                           Example 3:                           

Source:      sentía en mi alma el eco de la pregunta dirigida entonces a pedro: “¿me amas?
Translation: s en mia animo la econ de la demando direktita al pedro: ĉu vi amas min?
Reference:   mi sentis en mia animo la eĥon de la demando tiam adresita al petro: “ĉu vi amas min?
_____________________________________________________